Excelente pergunta. A sua série temporal é particularmente interessante porque ela representa o **fluxo de entrada de conteúdo no catálogo da Netflix**, e não uma série financeira tradicional. Isso abre várias possibilidades analíticas.

Pela figura, já é possível observar visualmente alguns padrões:

* **Tendência crescente** entre 2016 e 2019;
* **Maior volatilidade** a partir de 2017;
* **Picos extremos (outliers)**, especialmente no final de 2019/início de 2020;
* **Ausência aparente de sazonalidade anual forte**, embora isso precise ser testado formalmente;
* **Possível quebra estrutural** após 2020.

Abaixo estão as principais análises que você pode realizar.

---

# 1. Análise descritiva da série

Objetivo:

> Caracterizar o comportamento da entrada de filmes ao longo do tempo.

Perguntas:

* A Netflix aumentou a quantidade de filmes adicionados ao catálogo?
* Em quais períodos ocorreram os maiores volumes?
* Existem semanas atípicas?

Gráficos:

* Série temporal;
* Média móvel (4, 8 ou 12 semanas);
* Boxplot por ano;
* Heatmap ano × semana.

---

# 2. Decomposição da série temporal

Objetivo:

> Separar a série em componentes.

Modelo:

[
Y_t = T_t + S_t + R_t
]

onde:

* (T_t): tendência;
* (S_t): sazonalidade;
* (R_t): resíduo.

Com `statsmodels`:

```python
from statsmodels.tsa.seasonal import STL

stl = STL(
    week_added_movie.set_index('week_added')['count'],
    period=52
)

result = stl.fit()
result.plot()
```

Você descobrirá:

* se existe sazonalidade anual;
* qual é a tendência real;
* quando ocorreram choques.

---

# 3. Teste de tendência

Hipótese:

[
H_0:\ \text{não existe tendência}
]

[
H_1:\ \text{existe tendência}
]

Métodos:

### Mann-Kendall

Muito usado em séries ambientais.

```python
import pymannkendall as mk

mk.original_test(series)
```

---

# 4. Teste de estacionariedade

Objetivo:

> Verificar se a série é estacionária.

Muito importante para modelagem.

Teste ADF:

```python
from statsmodels.tsa.stattools import adfuller

adfuller(series)
```

Hipóteses:

[
H_0:\ \text{a série possui raiz unitária}
]

[
H_1:\ \text{a série é estacionária}
]

---

# 5. Mudanças estruturais

Pela figura, parece existir uma mudança próxima a **2020**.

Pergunta:

> O padrão de adição de filmes mudou após determinado período?

Métodos:

### Chow Test

(se você definir um ponto específico)

ou

### CUSUM

ou

### Change Point Detection

```python
import ruptures as rpt
```

Esse tipo de análise seria extremamente interessante para discutir:

* expansão global da Netflix;
* pandemia;
* mudança na estratégia de licenciamento.

---

# 6. Sazonalidade

Pergunta:

> Existem épocas do ano com maior número de filmes adicionados?

Pode-se analisar por:

## mês

```python
month = (
    netflix_movies
    .assign(month=lambda x: x['date_added'].dt.month)
    .groupby('month')
    .size()
)
```

ou

## semana do ano

```python
week = (
    netflix_movies
    .assign(
        week=lambda x:
        x['date_added'].dt.isocalendar().week
    )
    .groupby('week')
    .size()
)
```

Teste:

ANOVA ou Kruskal–Wallis.

---

# 7. Forecasting (previsão)

Pergunta:

> Quantos filmes esperaríamos nas próximas semanas?

Métodos:

### Simples

* Holt;
* Holt-Winters.

### Intermediários

* ARIMA;
* SARIMA.

### Avançados

* Prophet;
* LSTM.

---

# 8. Detecção de anomalias

Muito interessante para o seu gráfico.

Pergunta:

> Quais semanas apresentaram adições excepcionalmente altas?

Métodos:

### Z-score

```python
z = (
    series - series.mean()
) / series.std()
```

### IQR

### Isolation Forest

---

# 9. Comparação antes e depois da pandemia

Aparentemente muito promissora.

Hipóteses:

[
H_0:
\mu_{antes}=\mu_{depois}
]

[
H_1:
\mu_{antes}\neq\mu_{depois}
]

Exemplo:

* 2016–2019;
* 2020–2021.

Testes:

* t de Welch;
* Mann–Whitney.

---

# 10. Regressão temporal

Você pode modelar:

[
Filmes_t
========

\beta_0
+
\beta_1Tempo_t
+
\epsilon_t
]

ou

[
Filmes_t
========

\beta_0
+
\beta_1Tempo_t
+
\beta_2Pandemia_t
+
\epsilon_t
]

para quantificar:

* crescimento semanal;
* impacto de eventos específicos.

---

# Minha recomendação para um projeto exploratório da Netflix

Eu seguiria exatamente esta sequência:

1. **Visualização da série + média móvel**;
2. **Decomposição STL**;
3. **Teste de Mann–Kendall para tendência**;
4. **Detecção de mudanças estruturais (ruptures)**;
5. **Detecção de anomalias**;
6. **Comparação pré e pós-pandemia**;
7. **Modelo SARIMA para previsão**.

Essa combinação gera uma narrativa muito forte:

> *"O catálogo de filmes da Netflix apresentou crescimento significativo nas adições semanais até 2019, seguido por mudanças estruturais importantes. A decomposição temporal evidencia a predominância da tendência sobre a sazonalidade, enquanto a análise de anomalias identifica semanas de expansão excepcional do catálogo."*

Para um projeto de portfólio em Ciência de Dados, eu diria que as **três análises com maior valor** são:

* **Decomposição STL**;
* **Change Point Detection**;
* **Forecasting com SARIMA/Prophet**.

Elas mostram domínio simultâneo de **EDA, inferência estatística e modelagem de séries temporais**.
